In the following discussion, we will equate a Pokemon with its type, stats, and four known moves.  There's more to it, of course, but this seems like a good place to start.

# To Dos
- Solve the forme issue regarding stats

# The Problem

A pokemon's type, stats, and four known moves gives 12 dimensions of information about that pokemon.  While Player 1 win probability should be a nondecreasing function of the numeric stats, the amount of benefit provided from increasing a stat by one point depends on the matchup.  If you replace Weavile by Chien-Pao, your win probability should go up!  But how much it goes up should depend on the matchup.  If your opponent has no pokemon between Weavile's base 125 speed and Chien-Pao's base 135 speed, then your win probability shouldn't go up by very much.  Or, if your oppenent has lots of pokemon with high physical defense, then Chien-Pao's power increase (coming from its ability, not its stats) may not make much of a difference either.  But if your opponent has Eternatus (base speed 130), then replacing Weavile by Chien-Pao should make a big difference!  So a statistic's contribution to win probability is matchup dependent, and this seems hard for a model to learn.

Another issue is that for nearly every pokemon (maybe we should quantify this claim), their secondary attacking stat is irrelevant.  It does not matter what your Chien-Pao's special attack stat is; it will not affect the battle at all.  So when we're training a model, the model should not be told Chien-Pao's special attack stat.  One possibility is to replace every pokemon's attack and special attack stat with offensive_stat = max(atk, spa) and record a boolean offensive_stat_is_phys.  This is okay, but it seems like the model might have a hard time learning that if offensive_stat_is_phys == True, then offensive_stat should be compared with its opponents def stats, not spd stats.

Yet another issue is that the model will have to somehow divine the type chart from these battle outcomes.  This isn't a situation where having more Psychic types is better than having fewer; once again, it's matchup-dependent.

# A Solution

Rather than giving the model a bunch of basic information (types, stats, known moves) and hoping that the model learns the stuff we already know (Fire > Grass > Water > Fire), let's tell the model the stuff that we already know.  But how do you feed in the type chart and whatnot into the model?  Seems hard.

Instead, let's bake all of our knowledge into a set of offensive and defensive stats (we'll call these "advantage" stats) that are derived for each specific battle.  Then, we'll replace the basic stats with these "advantage" stats.

We will start off by neglecting the movepool.  We could try to map each move to its category, type, and base power, but that could be a lot of work.  It's something to explore in the future.

## Damage Approximator

The advantage stat starts with an approximation of damage.  Given a pokemon mon1 on team 1 and a pokemon mon2 on team 2, we can estimate the average damage that mon1 would do to mon2 by selecting its best STAB move (or a not-very-effective coverage move in the event that is better--this could be updated if we get data suggesting that in the event that a mon's best move is a coverage move, the coverage move is often neutral or better).  Note that we assume a base power of 80 for all moves.

We approximate that when mon1 attacks mon2, mon2 will lose the following fraction of its HP:

$$\text{damage}(\text{mon1},\text{mon2}) := \left(\frac{\left(\frac{2 \cdot \text{mon1}_{\text{level}}}{5} + 2\right) \cdot 80 \cdot \frac{\text{mon1}_{\text{off-stat}}}{\text{mon2}_{\text{def-stat}}}}{50} + 2\right) \cdot \text{type-multiplier}(\text{mon1},\text{mon2}) \cdot 92.5/100 / \text{mon2}_{\text{hp}}$$

where 

$$\text{mon1}_{\text{off-stat}} = \max(\text{mon1}_{\text{atk}}, \text{mon1}_{\text{spa}}),$$

$$ \text{mon2}_{\text{def-stat}} = \begin{cases} \text{mon2}_{\text{def}} & \text{if } \text{mon1}_{\text{off-stat}} = \text{mon1}_{\text{atk}} \\ \text{mon2}_{\text{spd}} & \text{else} \end{cases}, $$

$$ \text{type-multiplier}(\text{mon1},\text{mon2}) = \max\left(\frac{1}{2}, \max_{\text{type1} \in \text{mon1}_{\text{types}}} 1.5 \cdot \prod_{\text{type2} \in \text{mon2}_{\text{types}}} \text{eff}(\text{type1},\text{type2})\right) $$

with $\text{eff}(\text{type1},\text{type2})$ being determined by the type chart, so that, e.g. $\text{eff}(\text{water},\text{fire}) = 2$ while $\text{eff}(\text{fire},\text{water}) = 1/2$.

Some notes:
- This fraction is allowed to exceed 1 as the amount by which it exceeds 1 may actually matter (think: reflect/light screen/aurora veil or resistance berries).
- This uses a simplified version of the damage formula found here: https://bulbapedia.bulbagarden.net/wiki/Damage#Generation_V_onward
- The number 80 corresponds to the base power of the move theoretically being selected.
- The use of 92.5/100 corresponds to the average value of the 'random' factor.
- The use of type-multiplier is meant to approximate the product of STAB and Type.
- type-multiplier assumes that mon1 is only using STABs.  This could be updated to account for coverage moves in a future iteration on this stat.
- The max(1/2, stuff) in type-multiplier is to prevent type-multiplier from ever being 0.  It is very rare (though it does happen) that mon1 will be unable to damage mon2.  The factor of 1/2 is used because that is a multiplier for a not-very-effective coverage move.  This would be resolved by replacing the max over mon1 types by a max over mon1 move types.
- Damage or speed boosting items will not be accounted for.  This could be resolved in an ad-hoc way by checking for common boosting items (choice items, life orb).  It could also be resolved in a systemic way be using the smogon damage calculator to replace the offensive advantage stat.
- Damage/stat modifying abilities like Levitate, Thick Fat, or Sword of Ruin will not be accounted for.  This could 'only' be resolved by using the smogon damage calculator to replace this offensive advantage stat.
- Type chart modifying moves like Freeze-Dry are not accounted for.


## Advantage

The only (relevant) thing that doesn't go into the damage approximator is speed.  Speed is difficult to incorporate into advantage.  There are a few reasons for this:
  1. The only important feature of speed differential (meaning $\text{mon1}_{\text{spe}} - \text{mon2}_{\text{spe}}$) is its sign.  Magnitude is meaningless here.  So multiplying the damage approximation by speed differential would be a bad idea.
  2. The impact of speed differential can be large or small.  If you consider a hypothetical Weavile versus Iron Boulder matchup, each has a super-effective STAB on the other (meaning it has a type-multiplier equal to 3)!  In that situation, Weavile has the advantage because it goes first.  However, if you consider a Weavile versus Swampert matchup (where each has a type-multiplier of 1.5), the Swampert has the advantage in spite of its speed disadvantage due to its overall bulk.  My initial thought is that speed matters a lot when both pokemon are doing about the same amount of damage to one another, but doesn't matter very much when the pokemon are doing very different amounts of damage.  So 'having a speed advantage' should not correspond to a constant factor.

Also worth noting is that advantage depends not just on how much damage you're doing to your opponent, but how much damage your opponent is doing to you!

Maybe try computing 'turns to KO' for each mon and look at differential.  So we get something like

$$ \text{time-to-ko-diff}(\text{mon1},\text{mon2}) = \lceil \frac{1}{\min(1,\text{damage-approx}(\text{mon2},\text{mon1}))}\rceil - \lceil\frac{1}{\min(1,\text{damage-approx}(\text{mon1},\text{mon2}))}\rceil $$

Here, bigger is better for mon1.

Properties that I want:
- There should be a nice relationship between adv(mon1,mon2) and adv(mon2,mon1) (a + b = 1 with 0 < a,b < 1?  ab = 1?)
- If time-to-ko-diff is close to 0 and the times to ko are close to 1, the faster mon should have a big advantage (this is the situation where the faster mon just OHKOs the slower mon with no cost)
- If the time-to-ko-diff is close to 0 and the times to ko are large, the faster mon should have a small advantage (this is the situation where the faster mon KOs the slower mon but at an HP cost)
- If the time-to-ko-diff is large, the mon with the smaller time to ko should have a big advantage (this is the situation where one mon just dominates the other)

So maybe the advantage stat should represent something like: expected damage dealt to opponent in a 1v1 matchup?  If we let $n$ denote the round number in which the KO occurs, then the faster mon gets to go $n$ times and the slower mon gets to go $n$ or $n-1$ times depending on who wins.

Let's set

$$\text{time-to-ko}(\text{mon1},\text{mon2}) = \lceil \frac{1}{\min(1,\text{damage-approx}(\text{mon1},\text{mon2}))} \rceil$$

Then

$$\text{turn-of-ko}(\text{mon1},\text{mon2}) = \min\left(\text{time-to-ko}(\text{mon1},\text{mon2}), \text{time-to-ko}(\text{mon2},\text{mon1})\right)$$

So

$$
\text{1v1-damage}(\text{mon1},\text{mon2}) =
\begin{cases}
    (\text{turn-of-ko}(\text{mon1},\text{mon2}) - 1) \cdot \text{damage-approx}(\text{mon1},\text{mon2})  &\text{if } \text{mon1}_{\text{spe}} < \text{mon2}_{\text{spe}} \text{ and mon2 KOs mon1}\\
    \text{turn-of-ko}(\text{mon1},\text{mon2}) \cdot \text{damage-approx}(\text{mon1},\text{mon2})        &\text{else}
    
\end{cases}
$$

Then we can do something like set $\text{adv}(\text{mon1},\text{mon2}) = \text{1v1-damage}(\text{mon1},\text{mon2})$ or we can do something fancy and make it symmetric like $\text{adv}(\text{mon1},\text{mon2}) = \text{1v1-damage}(\text{mon1},\text{mon2}) - \text{1v1-damage}(\text{mon2},\text{mon1})$.

Regardless, this should be enough to get started.


## Potential problems with advantage stats

Some pokemon are not good because of their stats.  Take Sableye for example.  It has atrocious stats, but can win a match on the strength of its ability, Prankster.  These advantage stats won't account for that.  (On the other hand, neither will training on 12-dimensional info above.)

Other pokemon don't rely on their offensive stats for damage (think Toxapex).

Yet more pokemon rely heavily on priority moves.

# Testing

In order to test the FullPokemon class, we need:

- type_multiplier:
  - m1 has a 4x effective STAB (m1 = Weavile, m2 = Salamence)
  - m1 has a 2x effective STAB (m1 = Weavile, m2 = Haxorus)
  - m1's best STAB is neutral (m1 = Weavile, m2 = Corviknight)
  - m1's best STAB is 1/2x effective (m1 = Weavile, m2 = Chien-Pao)
  - m1's best STAB is 1/4x effective (m1 = Conkeldurr, m2 = Fezandipiti)
  - m1's best STAB is 0x effective (m1 = Banette, m2 = Wigglytuff)

- damage
  - already tested manually by comparing physical and special attackers on the calcs at calc.pokemonshowdown.com

- one_v_one_damage
  - m1 is faster than m2 and KOs m2 (m1 = Weavile, m2 = Salamence)
  - m2 is faster than m1 and KOs m1 (m1 = Sinistcha?, m2 = Weavile)
  - m1 is faster than m2 yet is KOd by m2 (m1 = Weavile, m2 = Conkeldurr)
  - m2 is faster than m1 yet is KOd by m1 (m1 = Swampert, m2 = Corviknight?)
  - m1 and m2 have a speed tie (m1 = Lanturn, m2 = Toxtricity)

In [40]:
import sys
import os
import json
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import copy
import itertools
import statsmodels.api as sm
from pathlib import Path
from sklearn.model_selection import StratifiedKFold,cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss,accuracy_score,roc_auc_score,confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
sys.path.insert(0,os.path.abspath('..'))
from tools import battle as b
from tools import full_pokemon as f

log_dir_1 = Path("../data/replays/gen9-randombattle")
log_dir_2 = Path("../data/replays/gen9-randombattle_2")
log_dir_3 = Path("../data/replays/gen9-randombattle_3")
log_dirs = [log_dir_1,log_dir_2,log_dir_3]
stat_names = ['hp','atk','def','spa','spd','spe']

In [2]:
desired_mon_names = ['Weavile','Salamence','Haxorus','Corviknight','Chien-Pao','Conkeldurr','Fezandipiti','Banette','Wigglytuff','Sinistcha','Swampert','Lanturn','Toxtricity','Ditto','Palafin','Terapagos','Regigigas']
desired_item_names = ["Light Ball","Choice Band","Choice Specs","Choice Scarf","Eviolite","Assault Vest","Soul Dew"]
desired_team_dict = {}
for file in log_dir_1.iterdir():
    if file.is_file():
        with open(file,"r") as battle_json:
            battle = json.load(battle_json)
        for team_index in range(2):
            for mon in battle["teams_full"][team_index].keys():
                mon_is_desired = (mon in desired_mon_names)
                item_is_desired = (battle["teams_full"][team_index][mon]["item"] in desired_item_names)
                if mon_is_desired or item_is_desired:
                    desired_team_dict[mon] = battle["teams_full"][team_index][mon]
                    if mon_is_desired:
                        desired_mon_names.remove(mon)
                    if item_is_desired:
                        desired_item_names.remove(battle["teams_full"][team_index][mon]["item"])
desired_mon_names = ['Weavile','Salamence','Haxorus','Corviknight','Chien-Pao','Conkeldurr','Fezandipiti','Banette','Wigglytuff','Sinistcha','Swampert','Lanturn','Toxtricity','Ditto','Palafin','Terapagos','Regigigas']
desired_item_names = ["Light Ball","Choice Band","Choice Specs","Choice Scarf","Eviolite","Assault Vest","Soul Dew"]


In [3]:
test_pokemon = {mon:f.FullPokemon(desired_team_dict[mon]) for mon in desired_team_dict.keys()}

In [4]:
test_pokemon["Weavile"].moves

['tripleaxel', 'lowkick', 'swordsdance', 'knockoff']

In [5]:
print("Testing Individual Pokemon")
assert(test_pokemon["Palafin"].stats["atk"] == 291), "Palafin failed"
assert(test_pokemon["Terapagos"].stats["spd"] == 214), "Terapagos failed"
assert(test_pokemon["Regigigas"].stat_multiplier("atk") == 0.5), "Regigigas atk failed"
assert(test_pokemon["Regigigas"].stat_multiplier("spa") == 1), "Regigigas spa failed"

print()
print("Testing items")
assert(test_pokemon["Kyogre"].stat_multiplier("spe") == 1.5), "Kyogre (Choice Scarf) spe failed"
assert(test_pokemon["Kyogre"].stat_multiplier("spa") == 1), "Kyogre (Choice Scarf) spa failed"
assert(test_pokemon["Greninja"].stat_multiplier("spa") == 1.5), "Greninja (Choice Specs) spa failed"
assert(test_pokemon["Greninja"].stat_multiplier("spe") == 1), "Greninja (Choice Specs) spe failed"
assert(test_pokemon["Azumarill"].stat_multiplier("atk") == 3), "Azumarill (Choice Band) atk failed"
assert(test_pokemon["Azumarill"].stat_multiplier("spe") == 1), "Azumarill (Choice Band) spe failed"
assert(test_pokemon["Banette"].damage_multiplier() == 5324/4096), "Banette (Life Orb) failed"
assert(test_pokemon["Pikachu"].stat_multiplier("spa") == 2), "Pikachu (Light Ball) spa failed"
assert(test_pokemon["Pikachu"].stat_multiplier("def") == 1), "Pikachu (Light Ball) def failed"
assert(test_pokemon["Chansey"].stat_multiplier("def") == 1.5), "Chansey (Eviolite) def failed"
assert(test_pokemon["Chansey"].stat_multiplier("spd") == 1.5), "Chansey (Eviolite) spd failed"
assert(test_pokemon["Chansey"].stat_multiplier("spe") == 1), "Chansey (Eviolite) spe failed"
assert(test_pokemon["Hoopa"].stat_multiplier("spd") == 1.5), "Hoopa (Assault Vest) spd failed"
assert(test_pokemon["Hoopa"].stat_multiplier("def") == 1), "Hoopa (Assault Vest) def failed"
assert(test_pokemon["Latias"].damage_multiplier() == 4915/4096), "Latias (Soul Dew) failed"

print()
print("Now testing type_multiplier")
assert(f.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Salamence']) == 6),"Weavile and Salamence failed"
assert(f.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Haxorus']) == 3), "Weavile and Haxorus failed"
assert(f.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Corviknight']) == 1.5), "Weavile and Corviknight failed"
assert(f.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Chien-Pao']) == 0.75), "Weavile and Chien-Pao failed"
assert(f.FullPokemon.type_multiplier(test_pokemon['Conkeldurr'],test_pokemon['Fezandipiti']) == 1/2), "Conkeldurr and Fezandipiti failed"
assert(f.FullPokemon.type_multiplier(test_pokemon['Banette'],test_pokemon['Wigglytuff']) == 1/2), "Banette and Wigglytuff failed"

print()
print("Now testing one_v_one_damage")
ovo = f.FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Salamence'])
assert(ovo[0] >= 1 and ovo[1] == 0), "Weavile and Salamence failed"
ovo = f.FullPokemon.one_v_one_damage(test_pokemon['Sinistcha'],test_pokemon['Weavile'])
assert(ovo[0] > 0 and ovo[0] < 1 and ovo[1] >= 1), "Sinistcha and Weavile failed"
ovo = f.FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Conkeldurr'])
assert(ovo[0] > 0 and ovo[0] < 1 and ovo[1] >=1),"Weavile and Conkeldurr failed"
ovo = f.FullPokemon.one_v_one_damage(test_pokemon['Swampert'],test_pokemon['Corviknight'])
assert(ovo[0] >= 1 and ovo[1] > 0 and ovo[1] < 1), "Swampert and Corviknight failed"
# ad hoc forcing Toxtricity to carry choice specs
test_pokemon["Toxtricity"].item = "Choice Specs"
ovo = f.FullPokemon.one_v_one_damage(test_pokemon["Lanturn"],test_pokemon["Toxtricity"])
assert(ovo[0] >= 1 and ovo[1] >= 1), "Lanturn and Toxtricity failed"

print()
print("Now testing Ditto")
ditto_dam_to_ban = f.FullPokemon.damage(test_pokemon["Ditto"],test_pokemon["Banette"])
ban_dam_to_ditto = f.FullPokemon.damage(test_pokemon["Banette"],test_pokemon["Ditto"])
ovo = f.FullPokemon.one_v_one_damage(test_pokemon["Ditto"],test_pokemon["Banette"])
assert(ditto_dam_to_ban > 0.84 and ditto_dam_to_ban < 1.01), "Ditto damage to Banette failed"
assert(ban_dam_to_ditto > 1.42 and ban_dam_to_ditto < 1.68), "Banette damage to Ditto failed"
assert(ovo[0] > 0.84 and ovo[0] < 1.01 and ovo[1] > 1), "Ditto and Banette failed"

Testing Individual Pokemon

Testing items

Now testing type_multiplier

Now testing one_v_one_damage

Now testing Ditto


## Examples

In [6]:
id = "2631906096"
with open("../data/replays/gen9-randombattle/gen9randombattle-" + id + ".json","r") as battle_json:
    data = json.load(battle_json)

team1 = [f.FullPokemon(data["teams_full"][0][key]) for key in data["teams_full"][0].keys()]
team2 = [f.FullPokemon(data["teams_full"][1][key]) for key in data["teams_full"][1].keys()]
col_names = [f"adv_over_{mon2.name}" for mon2 in team2]
col_names.insert(0,"p1_mon")
rows = []
for i,mon1 in enumerate(team1):
    rows.append([f.FullPokemon.advantage(mon1,mon2) for mon2 in team2])
    rows[i].insert(0,mon1.name)
df = pd.DataFrame(rows,columns=col_names)
adv_matrix = [[f.FullPokemon.advantage(mon1,mon2) for mon2 in team2] for mon1 in team1]
sum(sum(adv_matrix[i]) for i in range(len(adv_matrix)))

6.70447366467569

In [7]:
df

,p1_mon,adv_over_Abomasnow,adv_over_Ceruledge,adv_over_Chansey,adv_over_Hippowdon,adv_over_Carbink,adv_over_Klefki
0,Quaquaval,1.147629,1.123535,1.705675,0.858827,0.693693,0.314854
1,Pecharunt,1.050588,0.624728,0.185388,-0.674954,-0.312160,0.255685
2,Noivern,-0.756319,0.257875,-0.595860,0.900206,-0.794214,-0.862681
3,Azumarill,-1.031641,-0.066877,-0.713926,0.680872,0.350973,-0.343552
4,Gogoat,-0.768560,-1.037238,0.912017,0.872012,0.853608,-0.258471
5,Klawf,1.121944,1.654554,1.005346,-0.789109,0.170756,-1.030727


In [8]:
[[f"{adv_matrix[i][j]:.2f}" for j in range(6)] for i in range(6)]

[['1.15', '1.12', '1.71', '0.86', '0.69', '0.31'],
 ['1.05', '0.62', '0.19', '-0.67', '-0.31', '0.26'],
 ['-0.76', '0.26', '-0.60', '0.90', '-0.79', '-0.86'],
 ['-1.03', '-0.07', '-0.71', '0.68', '0.35', '-0.34'],
 ['-0.77', '-1.04', '0.91', '0.87', '0.85', '-0.26'],
 ['1.12', '1.65', '1.01', '-0.79', '0.17', '-1.03']]

## EDA

In [6]:
traits = [
        "trappers",
        "type_changers",
        "weather_setters",
        "terrain_setters",
        "stat_drop_resistors",
        "absorbers",
        "extra_immunities",
        "status_resists",
        "contact_punishers",
        "ability_ignorers",
        "pranksters",
        "intimidators",
        "unaware",
        "harvest",
        "regenerators",
        "serene_grace",
        "sturdy",
        "illusion",
        "triage",
        "boosting_abilities",
        "weather_boosters",
        "omni_boosters",
        "off_def_spe_boosters",
        "off_spe_boosters",
        "off_def_boosters",
        "off_boosters",
        "spe_boosters",
        "def_boosters",
        "move_boosters"
        ]

In [7]:
rows = []
for log_dir in log_dirs:
    for file in log_dir.iterdir():
        if file.is_file():
            battle = b.Battle(file,parse=True)
            if not battle.custom_ruleQ: # throw out battles with custom rules
                to_add = {
                    "id": battle.id,
                    "rated": battle.rated,
                    "duration": battle.end_time - battle.start_time,
                    "p1": battle.players[0],
                    "p2": battle.players[1],
                    "p1_rating" : battle.p1.elo0,
                    "elo_diff": battle.p1.elo0 - battle.p2.elo0,
                    "p1_wins" : battle.p1.name == battle.winner.name,
                    "p1_revealed_team_size" : len(battle.teams[0].keys()),
                    "p2_revealed_team_size" : len(battle.teams[1].keys()),
                }
                for trait in traits:
                    to_add["p1_num_" + trait] = 0
                    to_add["p2_num_" + trait] = 0
                # Team construction
                team1 = [f.FullPokemon(battle.teams_full[0][mon]) for mon in battle.teams_full[0].keys()]
                team2 = [f.FullPokemon(battle.teams_full[1][mon]) for mon in battle.teams_full[1].keys()]
                teams = [team1,team2]

                p1_types = set(mon.types[i] for mon in team1 for i in range(len(mon.types)))
                p2_types = set(mon.types[i] for mon in team2 for i in range(len(mon.types)))

                to_add["p1_type_diversity"] = len(p1_types)
                to_add["p2_type_diversity"] = len(p2_types)
                to_add["type_diversity_diff"] = len(p1_types) - len(p2_types)

                # stats and traits for each mon
                for player_index in range(2):
                    for mon_index in range(6):
                        mon = teams[player_index][mon_index]
                        to_add[f"p{player_index + 1}_m{mon_index + 1}_species_id"] = mon.speciesId
                        to_add[f"p{player_index + 1}_m{mon_index + 1}_type_1"] = mon.types[0]
                        try:
                            to_add[f"p{player_index + 1}_m{mon_index + 1}_type_2"] = mon.types[1]
                        except IndexError:
                            pass
                        for stat in stat_names:
                            to_add[f"p{player_index + 1}_m{mon_index + 1}_{stat}"] = mon.stats[stat]
                        if mon.is_trapper():
                            to_add[f"p{player_index + 1}_num_trappers"] += 1
                        if mon.is_type_changer():
                            to_add[f"p{player_index + 1}_num_type_changers"] += 1
                        if mon.is_weather_setter():
                            to_add[f"p{player_index + 1}_num_weather_setters"] += 1
                        if mon.is_terrain_setter():
                            to_add[f"p{player_index + 1}_num_terrain_setters"] +=1
                        if mon.is_stat_drop_resistor():
                            to_add[f"p{player_index + 1}_num_stat_drop_resistors"] += 1
                        if mon.is_absorber():
                            to_add[f"p{player_index + 1}_num_absorbers"] += 1
                        if mon.has_extra_immunities():
                            to_add[f"p{player_index + 1}_num_extra_immunities"] += 1
                        if mon.is_status_resistor():
                            to_add[f"p{player_index + 1}_num_status_resists"] += 1
                        if mon.is_contact_punisher():
                            to_add[f"p{player_index + 1}_num_contact_punishers"] += 1
                        if mon.is_ability_ignorer():
                            to_add[f"p{player_index + 1}_num_ability_ignorers"] += 1
                        if mon.ability == "Prankster":
                            to_add[f"p{player_index + 1}_num_pranksters"] += 1
                        if mon.ability == "Intimidate":
                            to_add[f"p{player_index + 1}_num_intimidators"] += 1
                        if mon.ability == "Unaware":
                            to_add[f"p{player_index + 1}_num_unaware"] += 1
                        if mon.ability == "Harvest":
                            to_add[f"p{player_index + 1}_num_harvest"] += 1
                        if mon.ability == "Regenerator":
                            to_add[f"p{player_index + 1}_num_regenerators"] += 1
                        if mon.ability == "Serene Grace":
                            to_add[f"p{player_index + 1}_num_serene_grace"] += 1
                        if mon.ability == "Sturdy":
                            to_add[f"p{player_index + 1}_num_sturdy"] += 1
                        if mon.ability == "Illusion":
                            to_add[f"p{player_index + 1}_num_illusion"] += 1
                        if mon.ability == "Triage":
                            to_add[f"p{player_index + 1}_num_triage"] += 1
                        if mon.has_boosting_ability():
                            to_add[f"p{player_index + 1}_num_boosting_abilities"] += 1
                        if mon.is_weather_booster():
                            to_add[f"p{player_index + 1}_num_weather_boosters"] += 1
                        if mon.has_omni_boost():
                            to_add[f"p{player_index + 1}_num_omni_boosters"] += 1
                        if mon.has_off_def_spe_boost():
                            to_add[f"p{player_index + 1}_num_off_def_spe_boosters"] += 1
                        if mon.has_off_spe_boost():
                            to_add[f"p{player_index + 1}_num_off_spe_boosters"] += 1
                        if mon.has_off_def_boost():
                            to_add[f"p{player_index + 1}_num_off_def_boosters"] += 1
                        if mon.has_off_boost():
                            to_add[f"p{player_index + 1}_num_off_boosters"] += 1
                        if mon.has_spe_boost():
                            to_add[f"p{player_index + 1}_num_spe_boosters"] += 1
                        if mon.has_def_boost():
                            to_add[f"p{player_index + 1}_num_def_boosters"] += 1
                        if mon.has_boost_move():
                            to_add[f"p{player_index + 1}_num_move_boosters"] += 1

                # advantage stats for each pair
                for p1_mon_index in range(6):
                    for p2_mon_index in range(6):
                        to_add[f"p1_m{p1_mon_index + 1}_adv_over_p2_m{p2_mon_index+1}"] = f.FullPokemon.advantage(team1[p1_mon_index],team2[p2_mon_index])

                rows.append(to_add)

# contains all information about all relevant matches in our data set
full_match_data = pd.DataFrame(rows)

for trait in traits:
    full_match_data[f"num_{trait}_diff"] = full_match_data[f"p1_num_{trait}"] - full_match_data[f"p2_num_{trait}"]

In [8]:
# The following lines are for grabbing subsets of features easily
type_col_names = [f"p{i+1}_m{m+1}_type_{t+1}" for i in range(2) for m in range(6) for t in range(2)]
stat_col_names = [f"p{p+1}_m{m+1}_{stat}" for p in range(2) for m in range(6) for stat in stat_names]
adv_col_names = [f"p1_m{i+1}_adv_over_p2_m{j+1}" for i in range(6) for j in range(6)]
trait_col_names = [f"num_{trait}_diff" for trait in traits]

In [9]:
# add the total advantage column for ease later
full_match_data["p1_total_adv"] = full_match_data[adv_col_names].sum(axis=1)

# initialize the reduced stat column names (replacing atk and spa by off)
red_stat_col_names = copy.deepcopy(stat_col_names)
# Add the offensive stat column and modify red_stat_col_names
for player_index in range(2):
    for mon_index in range(6):
        full_match_data[f"p{player_index + 1}_m{mon_index+1}_off"] = full_match_data[[f"p{player_index + 1}_m{mon_index + 1}_atk",f"p{player_index+1}_m{mon_index+1}_spa"]].max(axis=1)
        red_stat_col_names.remove(f"p{player_index + 1}_m{mon_index + 1}_atk")
        red_stat_col_names.remove(f"p{player_index + 1}_m{mon_index + 1}_spa")
        red_stat_col_names.append(f"p{player_index + 1}_m{mon_index + 1}_off")

red_stats = ['hp','off','def','spd','spe']
p1_tot = sum(full_match_data[f"p1_m{m+1}_{stat}"] for m in range(6) for stat in red_stats)
p2_tot = sum(full_match_data[f"p2_m{m+1}_{stat}"] for m in range(6) for stat in red_stats)
full_match_data["total_stat_diff"] = p1_tot - p2_tot

In [10]:
# These dataframes are for selecting out subsets of the data.
# We should throw away matches where people rage quit early
complete_matches = full_match_data[(full_match_data['duration'] > 60) & ((full_match_data["p1_revealed_team_size"] > 2) | (full_match_data["p2_revealed_team_size"] > 2))]
# This is just to easily grab matches where we know the players ratings
rated_matches = complete_matches[complete_matches['p1_rating'] > 0]
# This is to grab matches where we know that the players understand the basic switching strategy (from Marz' work on switching)
highly_rated_matches = rated_matches[(rated_matches['p1_rating'] > 1300) & (rated_matches[['p1_rating','elo_diff']].sum(axis=1) > 1300)]

In [11]:
highly_rated_matches

,id,rated,duration,p1,p2,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,...,p1_m4_off,p1_m5_off,p1_m6_off,p2_m1_off,p2_m2_off,p2_m3_off,p2_m4_off,p2_m5_off,p2_m6_off,total_stat_diff
1,gen9randombattle-2631763570,False,167,PineappleCats,L4V,1959,10,False,6,6,...,186,203,267,205,317,253,254,216,234,-10
2,gen9randombattle-2631369343,True,275,Chicken347,cococem,1999,-69,True,6,6,...,270,244,259,250,271,248,196,185,189,17
3,gen9randombattle-2631529004,False,123,WhatEver2102,Duck Cop,1999,17,True,1,3,...,288,250,243,239,236,176,239,232,206,-197
4,gen9randombattle-2631993792,False,301,monomythic,OverthereStair,2120,58,False,6,6,...,280,257,258,218,204,234,152,201,247,112
5,gen9randombattle-2631629401,False,116,Elite 4 Waally,jsjsiis,1958,38,False,5,4,...,219,301,255,182,235,183,269,217,188,-116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12761,gen9randombattle-2642081603,True,436,datboi480,throwaway20some,1933,-68,False,6,6,...,219,243,198,205,237,232,172,133,221,138
12762,gen9randombattle-2641877736,True,239,testbotcif,lay88,1325,-8,False,6,6,...,149,208,213,254,240,249,252,226,209,-186
12763,gen9randombattle-2642095210,True,187,notmetbh102,Uday30,2220,-49,False,6,5,...,288,199,187,191,250,228,221,227,198,103
12764,gen9randombattle-2642125538,True,186,Alienwere,forcemajor14,2023,4,False,6,5,...,171,249,246,223,258,239,240,172,217,-63


In [15]:
full_match_data['p1_wins'].value_counts(normalize=True)

p1_wins
False    0.52209
True     0.47791
Name: proportion, dtype: float64

In [16]:
highly_rated_matches['p1_wins'].value_counts(normalize=True)

p1_wins
False    0.523978
True     0.476022
Name: proportion, dtype: float64

In [17]:
full_match_data['elo_diff'].mean()

np.float64(-12.845370515431615)

In [18]:
highly_rated_matches['elo_diff'].mean()

np.float64(-14.925904088050315)

In [19]:
# sns.violinplot(full_match_data[['elo_diff','p1_wins']],x='elo_diff',hue='p1_wins')


In [20]:
# sns.violinplot(highly_rated_matches[['elo_diff','p1_wins']],x = 'elo_diff',hue='p1_wins')

In [21]:
#!{sys.executable} -m pip install fg-data-profiling

In [22]:
# import data_profiling
# from data_profiling import ProfileReport

In [23]:
# profile = ProfileReport(highly_rated_matches[['p1_rating','elo_diff','p1_total_adv'] + adv_col_names + ['p1_wins']],title = 'EDA_of_adv_in_highly_rated')
# profile.to_html()
# profile.to_file("advantage_in_highly_rated_matches_report.html")

In [24]:
# profile = ProfileReport(highly_rated_matches[["p1_total_adv","elo_diff"] + trait_col_names + ["p1_wins"]],title = "EDA_of_extra_traits")
# profile.to_html()
# profile.to_file("EDA_of_extra_traits.html")

In [25]:
highly_rated_matches

,id,rated,duration,p1,p2,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,...,p1_m4_off,p1_m5_off,p1_m6_off,p2_m1_off,p2_m2_off,p2_m3_off,p2_m4_off,p2_m5_off,p2_m6_off,total_stat_diff
1,gen9randombattle-2631763570,False,167,PineappleCats,L4V,1959,10,False,6,6,...,186,203,267,205,317,253,254,216,234,-10
2,gen9randombattle-2631369343,True,275,Chicken347,cococem,1999,-69,True,6,6,...,270,244,259,250,271,248,196,185,189,17
3,gen9randombattle-2631529004,False,123,WhatEver2102,Duck Cop,1999,17,True,1,3,...,288,250,243,239,236,176,239,232,206,-197
4,gen9randombattle-2631993792,False,301,monomythic,OverthereStair,2120,58,False,6,6,...,280,257,258,218,204,234,152,201,247,112
5,gen9randombattle-2631629401,False,116,Elite 4 Waally,jsjsiis,1958,38,False,5,4,...,219,301,255,182,235,183,269,217,188,-116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12761,gen9randombattle-2642081603,True,436,datboi480,throwaway20some,1933,-68,False,6,6,...,219,243,198,205,237,232,172,133,221,138
12762,gen9randombattle-2641877736,True,239,testbotcif,lay88,1325,-8,False,6,6,...,149,208,213,254,240,249,252,226,209,-186
12763,gen9randombattle-2642095210,True,187,notmetbh102,Uday30,2220,-49,False,6,5,...,288,199,187,191,250,228,221,227,198,103
12764,gen9randombattle-2642125538,True,186,Alienwere,forcemajor14,2023,4,False,6,5,...,171,249,246,223,258,239,240,172,217,-63


In [23]:
# normalize the elo_diff and p1_total_adv columns

ss = StandardScaler()

highly_rated_matches['normalized_elo_diff'] = ss.fit_transform(highly_rated_matches[['elo_diff']])
highly_rated_matches['normalized_adv'] = ss.fit_transform(highly_rated_matches[['p1_total_adv']]).copy()

/var/folders/0j/z6dzxfws7yld9pqcrrz48ll4001p55/T/ipykernel_30301/2843721259.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  highly_rated_matches['normalized_elo_diff'] = ss.fit_transform(highly_rated_matches[['elo_diff']])
/var/folders/0j/z6dzxfws7yld9pqcrrz48ll4001p55/T/ipykernel_30301/2843721259.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  highly_rated_matches['normalized_adv'] = ss.fit_transform(highly_rated_matches[['p1_total_adv']]).copy()


In [24]:
highly_rated_matches['normalized_elo_diff']

1        0.404533
2       -0.877590
3        0.518138
4        1.183544
5        0.858956
           ...   
12761   -0.861361
12762    0.112403
12763   -0.553002
12764    0.307156
12765   -0.358249
Name: normalized_elo_diff, Length: 10176, dtype: float64

In [ ]:
unbiased_model = sm.Logit(highly_rated_matches['p1_wins'],highly_rated_matches[['normalized_elo_diff','normalized_adv']],offset=0).fit()
biased_model = sm.Logit(highly_rated_matches['p1_wins'],highly_rated_matches[['elo_diff','p1_total_adv']]).fit()

Optimization terminated successfully.
         Current function value: 0.690274
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.689774
         Iterations 4


In [30]:
unbiased_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                p1_wins   No. Observations:                10176
Model:                          Logit   Df Residuals:                    10174
Method:                           MLE   Df Model:                            1
Date:                Wed, 08 Jul 2026   Pseudo R-squ.:                0.002490
Time:                        21:42:19   Log-Likelihood:                -7024.2
converged:                       True   LL-Null:                       -7041.8
Covariance Type:            nonrobust   LLR p-value:                 3.176e-09
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
normalized_elo_diff     0.0935      0.020      4.683      0.000       0.054       0.133
normalized_adv          0.1198      0.020      6.005      0.000       0.081       0.159
=======================================================================================
"""

In [31]:
biased_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                p1_wins   No. Observations:                10176
Model:                          Logit   Df Residuals:                    10174
Method:                           MLE   Df Model:                            1
Date:                Wed, 08 Jul 2026   Pseudo R-squ.:                0.003212
Time:                        21:42:24   Log-Likelihood:                -7019.1
converged:                       True   LL-Null:                       -7041.8
Covariance Type:            nonrobust   LLR p-value:                 1.741e-11
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
elo_diff         0.0018      0.000      5.697      0.000       0.001       0.002
p1_total_adv     0.0186      0.003      5.974      0.000       0.013       0.025
================================================================================
"""

In [39]:
biased_model.get_distribution()

In [43]:
biased_preds = list(map(round,(biased_model.predict(highly_rated_matches[['elo_diff','p1_total_adv']]))))
confusion_matrix(highly_rated_matches['p1_wins'],biased_preds,normalize='all')

array([[0.30945362, 0.21452437],
       [0.24695362, 0.2290684 ]])

In [45]:
unbiased_preds = list(map(round,unbiased_model.predict(highly_rated_matches[['normalized_elo_diff','normalized_adv']])))
confusion_matrix(highly_rated_matches['p1_wins'],unbiased_preds,normalize='all')

array([[0.27584513, 0.24813286],
       [0.21933962, 0.25668239]])

In [25]:
features = trait_col_names + ['type_diversity_diff','total_stat_diff']

models = [sm.Logit(highly_rated_matches['p1_wins'],highly_rated_matches[['normalized_elo_diff','normalized_adv',feature]],offset=0).fit(disp=False) for feature in features]

rows = []
for i in range(len(features)):
    summary = pd.Series({"feature" : features[i], "p-value" : models[i].pvalues.iloc[2], "coefficient" : models[i].params.iloc[2]})
    rows.append(summary)

table = pd.DataFrame(rows)
table.sort_values("p-value",ascending=True)

,feature,p-value,coefficient
26,num_spe_boosters_diff,0.038780,0.033195
20,num_weather_boosters_diff,0.068633,-0.063711
14,num_regenerators_diff,0.071210,0.070266
2,num_weather_setters_diff,0.084439,0.066437
30,total_stat_diff,0.093450,-0.000161
23,num_off_spe_boosters_diff,0.099249,0.034128
0,num_trappers_diff,0.125221,-0.105889
18,num_triage_diff,0.140238,0.169129
3,num_terrain_setters_diff,0.149629,0.080532
7,num_status_resists_diff,0.187560,-0.045147


In [27]:
skf = StratifiedKFold(n_splits=10,shuffle=True,random_state=207)

log_reg_elo_diff = LogisticRegression(penalty=None,max_iter=1000,n_jobs=-1)

accs = []
lls = []
roc_aucs = []
for train_ind,test_ind in skf.split(highly_rated_matches[['elo_diff']],highly_rated_matches['p1_wins']):
    log_reg_elo_diff.fit(highly_rated_matches[['elo_diff']].iloc[train_ind],highly_rated_matches['p1_wins'].iloc[train_ind])
    preds = log_reg_elo_diff.predict(highly_rated_matches[['elo_diff']].iloc[test_ind])
    pred_probas = log_reg_elo_diff.predict_proba(highly_rated_matches[['elo_diff']].iloc[test_ind])
    accs.append(accuracy_score(highly_rated_matches['p1_wins'].iloc[test_ind],preds))
    lls.append(log_loss(highly_rated_matches['p1_wins'].iloc[test_ind],pred_probas))
    roc_aucs.append(roc_auc_score(highly_rated_matches[['p1_wins']].iloc[test_ind],pred_probas[:,1]))

print("Baseline model (logistic regression on Elo differential):")
print(f"\t Accuracy mean = {np.mean(accs)} with standard deviation = {np.std(accs,ddof=1)}")
print(f"\t Log-loss mean = {np.mean(lls)} with standard deviation = {np.std(lls,ddof=1)}")
print(f"\t ROC-AUC mean = {np.mean(roc_aucs)} with standard deviation = {np.std(roc_aucs,ddof=1)}")

Baseline model (logistic regression on Elo differential):
	 Accuracy mean = 0.5291851877609133 with standard deviation = 0.010724634829718974
	 Log-loss mean = 0.6910203209814452 with standard deviation = 0.0015544084012582228
	 ROC-AUC mean = 0.5247629489416715 with standard deviation = 0.018882005029829008


/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning,

In [61]:
# for training on highly rated matches
skf = StratifiedKFold(n_splits=10,shuffle=True,random_state=207)
biased_log_reg = LogisticRegression(C=np.inf,max_iter=10000)
unbiased_log_reg = LogisticRegression(C=np.inf,max_iter=10000,fit_intercept=False)
rand_forest = RandomForestClassifier(max_depth = 4, min_samples_leaf=20, n_estimators=500,criterion='log_loss',random_state=207)
# new_rand_forest = RandomForestClassifier(max_depth = 4, min_samples_leaf=20, n_estimators=500,criterion='log_loss',n_jobs=-1)
# rand_forest_with_type = Pipeline([
#     ('one_hot', ColumnTransformer(
#         [('types', OneHotEncoder(handle_unknown='ignore'), type_col_names)],
#         remainder='passthrough'
#     )),
#     ('forest', RandomForestClassifier(n_estimators=100,criterion='log_loss',n_jobs=-1))
# ])

useful_traits = ["num_move_boosters_diff"]

models = [
    ('biased_adv_elo',biased_log_reg),
    ('unbiased_adv_elo',unbiased_log_reg),
    ('unbiased_adv_elo_traits',unbiased_log_reg),
    ('unbiased_adv_elo_stats_traits',unbiased_log_reg),
]

target = highly_rated_matches['p1_wins']
feature_sets = [
    highly_rated_matches[['p1_total_adv','elo_diff']],
    highly_rated_matches[['normalized_adv','normalized_elo_diff']],
    highly_rated_matches[['normalized_adv','normalized_elo_diff'] + useful_traits],
    highly_rated_matches[['normalized_adv','normalized_elo_diff','total_stat_diff'] + useful_traits],
]

scoring = {
    'accuracy' : 'accuracy',
    'log_loss' : 'neg_log_loss',
    'roc_auc' : 'roc_auc'
}

def summarize(name, cvres):
    return pd.Series({
        "model": name,
        "roc_auc_mean":  cvres["test_roc_auc"].mean(),
        "roc_auc_std":   cvres["test_roc_auc"].std(ddof=1),
        "log_loss_mean": -cvres["test_log_loss"].mean(),
        "log_loss_std":  (-cvres["test_log_loss"]).std(ddof=1),
        "acc_mean":      cvres["test_accuracy"].mean(),
        "acc_std":       cvres["test_accuracy"].std(ddof=1),
    })

rows = []
for ind in range(len(models)):
    model = models[ind][1]
    cv = cross_validate(model,feature_sets[ind],target,cv=skf,scoring=scoring,n_jobs=-1)
    rows.append(summarize(models[ind][0],cv))


print("Confusion Matrices:")
for i in range(len(models)):
    print(f"Confusion matrix for {models[i][0]}:")
    model = models[i][1]
    model.fit(feature_sets[i],highly_rated_matches['p1_wins'])
    cm = confusion_matrix(highly_rated_matches['p1_wins'],model.predict(feature_sets[i]),normalize='all')
    print(cm)
    print()

summary = pd.DataFrame(rows)
summary.sort_values("acc_mean",ascending=False)

Confusion Matrices:
Confusion matrix for biased_adv_elo:
[[0.3992728  0.12470519]
 [0.3351022  0.14091981]]

Confusion matrix for unbiased_adv_elo:
[[0.27584513 0.24813286]
 [0.21933962 0.25668239]]

Confusion matrix for unbiased_adv_elo_traits:
[[0.27584513 0.24813286]
 [0.21904481 0.2569772 ]]

Confusion matrix for unbiased_adv_elo_stats_traits:
[[0.2759434  0.24803459]
 [0.22002752 0.2559945 ]]



,model,roc_auc_mean,roc_auc_std,log_loss_mean,log_loss_std,acc_mean,acc_std
0,biased_adv_elo,0.542534,0.010615,0.689302,0.001620,0.539407,0.008072
3,unbiased_adv_elo_stats_traits,0.542577,0.009950,0.690448,0.001700,0.532820,0.011212
2,unbiased_adv_elo_traits,0.542028,0.010492,0.690520,0.001705,0.532034,0.008692
1,unbiased_adv_elo,0.542533,0.010614,0.690469,0.001585,0.531151,0.011918


In [24]:
# for training on extreme advantage matchups
# Let's train a model on extreme_adv_matchups, then have it predict highly_rated_matches.
# We'll train on 80% of extreme_adv_matchups, then predict on the 20% of eam and 20% of eam's complement in hrm (not exactly these numbers, unless n_splits = 5)

n_splits = 10
features = ["elo_diff","p1_total_adv"]

# determines what's extreme - 4.2 will call extreme the upper and lower quartile, while 10.4 will call extreme the upper and lower 5%
for threshold in [0.1,2,4,6,8,10,15]:

    extreme_adv_matchups = highly_rated_matches[abs(highly_rated_matches['p1_total_adv']) > threshold]
    small_adv_matchups = highly_rated_matches[abs(highly_rated_matches["p1_total_adv"]) <= threshold]

    skf = StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=207)
    log_reg = LogisticRegression(penalty=None,n_jobs=-1,max_iter=10000)

    eam_split = skf.split(extreme_adv_matchups[features],extreme_adv_matchups["p1_wins"])
    sam_split = skf.split(small_adv_matchups[features],small_adv_matchups["p1_wins"])

    eam_accs = []
    sam_accs = []
    total_accs = []

    for (eam_tt_ind,eam_val_ind),(sam_tt_ind,sam_val_ind) in itertools.zip_longest(eam_split,sam_split):
        log_reg.fit(extreme_adv_matchups[features].iloc[eam_tt_ind],extreme_adv_matchups["p1_wins"].iloc[eam_tt_ind])
        eam_preds = log_reg.predict(extreme_adv_matchups[features].iloc[eam_val_ind])
        sam_preds = log_reg.predict(small_adv_matchups[features].iloc[sam_val_ind])
        eam_acc = accuracy_score(extreme_adv_matchups["p1_wins"].iloc[eam_val_ind],eam_preds)
        sam_acc = accuracy_score(small_adv_matchups["p1_wins"].iloc[sam_val_ind],sam_preds)
        eam_num_correct = eam_acc * len(eam_preds)
        sam_num_correct = sam_acc * len(sam_preds)
        total_acc = (eam_num_correct + sam_num_correct) / (len(eam_preds) + len(sam_preds))
        eam_accs.append(eam_acc)
        sam_accs.append(sam_acc)
        total_accs.append(total_acc)
    print(f"At threshold {threshold},")
    print(f"\t the average total accuracy is {np.mean(total_accs)}")
    print(f"\t the average extreme match accuracy is {np.mean(eam_accs)}")
    print(f"\t the average small advantage accuracy is {np.mean(sam_accs)}")

At threshold 0.1,
	 the average total accuracy is 0.5397663854148118
	 the average extreme match accuracy is 0.5397044334975369
	 the average small advantage accuracy is 0.5435897435897437
At threshold 2,
	 the average total accuracy is 0.5405367464773082
	 the average extreme match accuracy is 0.5413863877118643
	 the average small advantage accuracy is 0.538034188034188
At threshold 4,
	 the average total accuracy is 0.5396604120021323
	 the average extreme match accuracy is 0.5410883092018666
	 the average small advantage accuracy is 0.5380290179156365
At threshold 6,
	 the average total accuracy is 0.5406389497286745
	 the average extreme match accuracy is 0.5473554590375477
	 the average small advantage accuracy is 0.5370647507618391
At threshold 8,
	 the average total accuracy is 0.5372397800699814
	 the average extreme match accuracy is 0.5454762609394157
	 the average small advantage accuracy is 0.5350252559935296
At threshold 10,
	 the average total accuracy is 0.5381977595582

In [20]:
extreme_adv_matchups

,id,rated,duration,p1,p2,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,...,p1_m3_off,p1_m4_off,p1_m5_off,p1_m6_off,p2_m1_off,p2_m2_off,p2_m3_off,p2_m4_off,p2_m5_off,p2_m6_off
2,gen9randombattle-2631369343,True,275,Chicken347,cococem,1999,-69,True,6,6,...,261,270,244,259,250,271,248,196,185,189
3,gen9randombattle-2631529004,False,123,WhatEver2102,Duck Cop,1999,17,True,1,3,...,180,288,250,243,239,236,176,239,232,206
4,gen9randombattle-2631993792,False,301,monomythic,OverthereStair,2120,58,False,6,6,...,190,280,257,258,218,204,234,152,201,247
5,gen9randombattle-2631629401,False,116,Elite 4 Waally,jsjsiis,1958,38,False,5,4,...,217,219,301,255,182,235,183,269,217,188
6,gen9randombattle-2631479578,False,112,gamergray23,TheOldTimers,1928,1,False,3,2,...,200,255,215,253,216,217,222,205,215,226
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12764,gen9randombattle-2641938445,True,277,andonibavi,qiuescent,2173,-43,True,5,5,...,253,141,198,202,318,232,168,232,254,197
12766,gen9randombattle-2642081603,True,436,datboi480,throwaway20some,1933,-68,False,6,6,...,254,219,243,198,205,237,232,172,133,221
12767,gen9randombattle-2641877736,True,239,testbotcif,lay88,1325,-8,False,6,6,...,228,149,208,213,254,240,249,252,226,209
12768,gen9randombattle-2642095210,True,187,notmetbh102,Uday30,2220,-49,False,6,5,...,317,288,199,187,191,250,228,221,227,198


Correlations at least 0.01:
0.045 (1 - elo_diff)
0.032 (4 - p1_num_boosting)
0.013 (5 - p1_num_contant_punishers)
0.013 (8 - p1_num_harvest)
0.011 (11 - p1_num_move_boosters)
0.014 (12 - p1_num_off_boosters)
0.018 (18 - p1_num_regenerators)
0.011 (22 - p1_status_resists new)
0.010 (26 - p1_num_triage)
0.019 (30 - p1_num_weather_setters) - consider adding manual setters to see if this boosts - no this was very bad, down to 0.09
0.051 (31 - p1_total_adv)
0.014 (38 - p2_num_extra_immunities)
0.013 (43 - p2_num_off_boosters)
0.013 (51 - p2_num_spe_boosters)
0.019 (52 - p2_num_stat_drop_resistors) - might be a bad category, consider separating out the good abilities from the useless - good change, down to 0
0.014 (53 - p2_num_status_resists) - might also be bad, consider separating out good from useless - also good change, down to 0
0.011 (56 - p2_num_trappers) - yep, trappers are bad in this format
0.015 (59 - p2_num_unaware) - what the heck

look at differential next


0.045 (1 - elo_diff)
0.013 (3 - absorbers_diff)
0.020 (4 - boosting_abilities_diff)
0.019 (11 - move_boosters_diff)
0.035 (12 - off_boosters_diff)
0.015 (14 - off_def_spe_boosters_diff)
0.013 (18 - regenerators_diff)
0.011 (25 - trappers_diff)
0.014 (28 - unaware_diff)
0.011 (30 - weather_setters_diff)



0.051 - total_adv
0.045 - elo_diff
0.035 - off_boosters_diff
0.032 - p1_num_boosting_abilities
0.020 - boosting_abilities_diff
0.019 - move_boosters_diff
0.019 - p1_num_weather_setters
0.018 - p1_num_regenerators
0.015 - off_def_spe_boosters_diff


total_adv
elo_diff
off_boosters_diff/move_boosters_diff/off_def_spe_boosters_diff
boosting_abilities_diff - consider tuning to filter out any obviously bad ones
weather_setters_diff
regenerators_diff
unaware_diff?
triage_diff?

